## Finetuning Gemma Model

First, we need to install the required libraries. This includes `transformers`, `accelerate`, `bitsandbytes`, `peft`, and `trl`.

In [ ]:
pip install -q -U transformers accelerate bitsandbytes peft trl datasets

Now, let's load the Gemma model and its tokenizer. We'll use a `GemmaForCausalLM` from the `transformers` library and load it in 4-bit quantization for efficient memory usage.

In [25]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata

# Get HF token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

model_id = "google/gemma-2b"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

print("Model and tokenizer loaded successfully!")

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Model and tokenizer loaded successfully!


In [26]:
text = "Quote: Imagination is more,"
device = "cuda:0"

inputs = tokenizer(text, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Quote: Imagination is more, than knowledge.

I am a self-taught artist, born in 1985 in the beautiful city of Porto, Portugal.

I have always been interested in art, but I never thought I would be able to make a living out


### 1. Prepare Dataset
We will use a sample dataset from Hugging Face for this demonstration. We also need to define a formatting function to structure the prompts for Gemma.

In [27]:
from datasets import load_dataset

dataset_name = "Abirate/english_quotes"
dataset = load_dataset(dataset_name, split="train")

def formatting_func(example):
    text = f"Quote: {example['quote']}\nAuthor: {example['author']}"
    return text

# Apply the formatting function to create a 'text' column
dataset = dataset.map(lambda x: tokenizer(formatting_func(x)))

print(f"Dataset loaded and formatted: {dataset_name}")
print(dataset)

Dataset loaded and formatted: Abirate/english_quotes
Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 2508
})


### Using a Custom Dataset

If you want to use your own data, upload your file (e.g., `my_data.json` or `my_data.csv`) to the Colab file browser on the left, then use the following code block to load it.

In [ ]:
from datasets import load_dataset

# --- CHOOSE YOUR FORMAT ---
# For JSON/JSONL:
# my_dataset = load_dataset('json', data_files='my_data.json', split='train')

# For CSV:
# my_dataset = load_dataset('csv', data_files='my_data.csv', split='train')

# 2. Update the formatting function based on YOUR column names
def custom_formatting_func(example):
    # If your file has columns named 'instruction' and 'response', change these:
    text = f"### Question: {example['question']}\n### Answer: {example['answer']}"
    return text

print("Code updated! The column names in your file can be anything; just match them in the function above.")

### 2. Configure LoRA
LoRA allows us to fine-tune only a small number of additional parameters, making it possible to train large models on consumer GPUs.

In [28]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

### 3. Training with SFTTrainer
We use `SFTTrainer` from the `trl` library, which simplifies the supervised fine-tuning process.

### 3. Initialize the SFTTrainer
We use the `SFTTrainer` to manage the training process. Note that we pass the `base_model` here instead of the previously modified `model` variable to ensure the trainer starts with a clean version of Gemma before applying the new `lora_config`.

In [34]:
import transformers
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=20,
        max_steps=500,
        learning_rate=1e-4,
        bf16=True,
        fp16=False,
        logging_steps=1,
        output_dir="outputs",
        optim="paged_adamw_8bit"
    ),
    peft_config=lora_config,
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [35]:
trainer.train()

Step,Training Loss
1,2.563384
2,1.917098
3,2.462584
4,2.812457
5,2.328629
6,2.502011
7,2.890218
8,2.264982
9,3.143750
10,2.472627


Step,Training Loss
1,2.563384
2,1.917098
3,2.462584
4,2.812457
5,2.328629
6,2.502011
7,2.890218
8,2.264982
9,3.143750
10,2.472627


TrainOutput(global_step=500, training_loss=1.721113628745079, metrics={'train_runtime': 1562.6071, 'train_samples_per_second': 1.28, 'train_steps_per_second': 0.32, 'total_flos': 1177543037214720.0, 'train_loss': 1.721113628745079, 'epoch': 0.7974481658692185})

In [38]:
trainer.model.save_pretrained("my_lora_model")
tokenizer.save_pretrained("my_lora_model")

('my_lora_model/tokenizer_config.json', 'my_lora_model/tokenizer.json')

In [39]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model_name = "google/gemma-2b"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.float16,
    device_map="auto"
)

print('Base model loaded successfully')

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Base model loaded successfully


In [36]:
pip install -q -U torchao

In [40]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    "my_lora_model"
)

In [41]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GemmaForCausalLM(
      (model): GemmaModel(
        (embed_tokens): GemmaTextScaledWordEmbedding(256000, 2048, padding_idx=0)
        (layers): ModuleList(
          (0-17): 18 x GemmaDecoderLayer(
            (self_attn): GemmaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj):

In [43]:
prompt = "Quote: Be yourself; everyone else is already taken."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=50
)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)


Quote: Be yourself; everyone else is already taken.
Author: Oscar Wilde, The Picture of Dorian Gray
Author: Oscar Wilde, The Picture of Dorian Gray
Author: Oscar Wilde, The Picture of Dorian Gray
Author: Oscar Wilde, The Picture of Dorian Gray
Author: Oscar Wilde,


### Adapting for Other Models

The code above is modular. To use a different model, update the `model_id` and ensure the LoRA `target_modules` match the architecture of your chosen model.

In [ ]:
# Example for adapting to a Mistral or Llama model
# model_id = "mistralai/Mistral-7B-v0.1"

# Most modern models use these target modules:
# lora_config = LoraConfig(
#     r=8,
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#     task_type="CAUSAL_LM",
# )

print("To switch models, update the model_id variable at the top and re-run the cells.")